In [1]:
from tensorflow.keras.applications import EfficientNetB3, MobileNetV2, ResNet50, VGG19, DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Concatenate, Input
from tensorflow.keras.models import Model

NUM_CLASSES = 5

def build_base(model):
    x = model.output
    x = GlobalAveragePooling2D()(x)
    return Model(model.input, x)

def get_models():
    base_models = [
        EfficientNetB3(weights='imagenet', include_top=False, input_shape=(224,224,3)),
        MobileNetV2(weights='imagenet', include_top=False),
        ResNet50(weights='imagenet', include_top=False),
        VGG19(weights='imagenet', include_top=False),
        DenseNet121(weights='imagenet', include_top=False)
    ]
    return [build_base(m) for m in base_models]

def DFEN():
    inputs = Input(shape=(224,224,1))

    models = get_models()

    features = [m(inputs) for m in models]

    # FEATURE FUSION
    fused = Concatenate()(features)

    x = Dense(256, activation='relu')(fused)
    x = Dense(128, activation='relu')(x)

    output = Dense(NUM_CLASSES, activation='softmax')(x)

    return Model(inputs, output)

In [2]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Concatenate, Input
from tensorflow.keras.layers import Activation, BatchNormalization
from tensorflow.keras.models import Model

# =========================================
# FIRE MODULE (Core of SqueezeNet)
# =========================================
def fire_module(x, squeeze_filters, expand_filters):

    # Squeeze layer (1x1 conv)
    squeeze = Conv2D(squeeze_filters, (1,1), padding='valid', activation='relu')(x)

    # Expand layer (1x1 + 3x3 conv)
    expand_1x1 = Conv2D(expand_filters, (1,1), padding='valid', activation='relu')(squeeze)
    expand_3x3 = Conv2D(expand_filters, (3,3), padding='same', activation='relu')(squeeze)

    # Concatenate expand outputs
    x = Concatenate(axis=-1)([expand_1x1, expand_3x3])

    return x

# =========================================
# CUSTOM SQUEEZENET MODEL
# =========================================
def SqueezeNet(input_shape=(224,224,3), num_classes=5):

    inputs = Input(shape=input_shape)

    # Initial convolution
    x = Conv2D(96, (7,7), strides=2, padding='same', activation='relu')(inputs)
    x = MaxPooling2D(pool_size=(3,3), strides=2)(x)

    # Fire modules
    x = fire_module(x, 16, 64)
    x = fire_module(x, 16, 64)
    x = MaxPooling2D(pool_size=(3,3), strides=2)(x)

    x = fire_module(x, 32, 128)
    x = fire_module(x, 32, 128)
    x = MaxPooling2D(pool_size=(3,3), strides=2)(x)

    x = fire_module(x, 48, 192)
    x = fire_module(x, 48, 192)
    x = fire_module(x, 64, 256)

    # Final layers
    x = Conv2D(num_classes, (1,1), activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    outputs = Activation('softmax')(x)

    model = Model(inputs, outputs, name="SqueezeNet_Custom")

    return model

In [5]:
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Concatenate,
                                     GlobalAveragePooling2D, Conv2D,
                                     MaxPooling2D)
from tensorflow.keras.models import Model

from tensorflow.keras.applications import (
    EfficientNetB3, MobileNetV2, ResNet50, VGG19, DenseNet121
)

NUM_CLASSES = 5

# =========================================================
# SQUEEZENET (FIXED - FEATURE OUTPUT MODE)
# =========================================================
def fire_module(x, squeeze, expand):

    squeeze_layer = Conv2D(squeeze, (1,1), activation='relu', padding='valid')(x)

    expand1 = Conv2D(expand, (1,1), activation='relu', padding='valid')(squeeze_layer)
    expand3 = Conv2D(expand, (3,3), activation='relu', padding='same')(squeeze_layer)

    return Concatenate()([expand1, expand3])


def SqueezeNet(input_shape=(224,224,3)):

    inputs = Input(shape=input_shape)

    x = Conv2D(96, (7,7), strides=2, activation='relu', padding='same')(inputs)
    x = MaxPooling2D((3,3), strides=2)(x)

    x = fire_module(x, 16, 64)
    x = fire_module(x, 16, 64)
    x = MaxPooling2D((3,3), strides=2)(x)

    x = fire_module(x, 32, 128)
    x = fire_module(x, 32, 128)
    x = MaxPooling2D((3,3), strides=2)(x)

    x = fire_module(x, 48, 192)
    x = fire_module(x, 48, 192)

    # IMPORTANT: return FEATURE MAP (NOT CLASS OUTPUT)
    x = GlobalAveragePooling2D()(x)

    return Model(inputs, x, name="SqueezeNet")

# =========================================================
# FEATURE EXTRACTOR MODELS
# =========================================================
def get_models(input_shape=(224,224,3)):

    return [
        EfficientNetB3(include_top=False, weights='imagenet', input_shape=input_shape),
        MobileNetV2(include_top=False, weights='imagenet', input_shape=input_shape),
        ResNet50(include_top=False, weights='imagenet', input_shape=input_shape),
        VGG19(include_top=False, weights='imagenet', input_shape=input_shape),
        DenseNet121(include_top=False, weights='imagenet', input_shape=input_shape),
        SqueezeNet(input_shape)
    ]

# =========================================================
# DFEN MODEL (FIXED FEATURE SHAPES)
# =========================================================
def DFEN(input_shape=(224,224,3)):

    inputs = Input(shape=input_shape)

    base_models = get_models(input_shape)

    features = []

    for model in base_models:

        x = model(inputs)

        # SAFETY CHECK (ensures 4D → 2D conversion)
        if len(x.shape) == 4:
            x = GlobalAveragePooling2D()(x)

        features.append(x)

    # FEATURE FUSION
    fused = Concatenate()(features)

    x = Dense(256, activation='relu')(fused)
    x = Dense(128, activation='relu')(x)

    outputs = Dense(NUM_CLASSES, activation='softmax')(x)

    return Model(inputs, outputs)

# =========================================================
# BUILD & TEST
# =========================================================
model = DFEN()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetb3      │ (None, 7, 7,      │ 10,783,535 │ input_layer_7[0]… │
│ (Functional)        │ 1536)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mobilenetv2_1.00_2… │ (None, 7, 7,      │  2,257,984 │ input_layer_7[0]… │
│ (Functional)        │ 1280)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ input_layer_7[0]… │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg19 (Functional)  │ (None, 7, 7, 512) │ 20,024,384 │ input_layer_7[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ densenet121         │ (None, 7, 7,      │  7,037,504 │ input_layer_7[0]… │
│ (Functional)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1536)      │          0 │ efficientnetb3[0… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ mobilenetv2_1.00… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg19[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1024)      │          0 │ densenet121[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ SqueezeNet          │ (None, 384)       │    349,248 │ input_layer_7[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_12      │ (None, 6784)      │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ global_average_p… │
│                     │                   │            │ global_average_p… │
│                     │                   │            │ global_average_p… │
│                     │                   │            │ global_average_p… │
│                     │                   │            │ SqueezeNet[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │  1,736,960 │ concatenate_12[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     32,896 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 5)         │        645 │ dense_2[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 65,810,868 (251.05 MB)

 Trainable params: 65,552,685 (250.06 MB)

 Non-trainable params: 258,183 (1008.53 KB)